In [ ]:
#!/usr/bin/env python3

import requests
from bs4 import BeautifulSoup
import time
import random
import json
from urllib.parse import urljoin
# ------------------ GLOBAL SETTINGS ------------------

MAX_ARTICLES = 1000
OUT_FILE = "news_test_punjabi.jsonl"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (AcademicResearchBot/1.0)",
    "Accept-Language": "te-IN,ta-IN,hi-IN,bn-IN,en;q=0.8"
}

MIN_DELAY = 1.0
MAX_DELAY = 2.0


# ------------------ NEWSPAPER CONFIG ------------------

NEWSPAPERS = {

    # ---------------- KANNADA ----------------
"kannada_wikipedia": {
    "source": "wikipedia-kn",
    "base_url": "https://kn.wikipedia.org/wiki/ವರ್ಗ:ಭಾರತ",
    "article_pattern": "/wiki/",
    "language": "kn",
    "section": "india",
    "stop_markers": [
        "ಉಲ್ಲೇಖಗಳು",
        "ಬಾಹ್ಯ ಕೊಂಡಿಗಳು",
        "ಇನ್ನೂ ನೋಡಿ",
        "ವಿಕಿಪೀಡಿಯ"
    ]
},


    # ---------------- ODIA ----------------
"odia_ommcomnews": {
    "source": "ommcom-or",
    "base_url": "https://ommcomnews.com/category/national/",
    "article_pattern": "/national/",
    "language": "or",
    "section": "india",
    "stop_markers": [
        "ଅଧିକ ପଢ଼ନ୍ତୁ",
        "Advertisement",
        "ବିଜ୍ଞାପନ",
        "©",
        "Ommcom News",
        "ସବସ୍କ୍ରାଇବ",
        "ଲାଇଭ୍"
    ]
},

"malayalam_deshabhimani": {
        "source": "deshabhimani",
        "base_url": "https://www.deshabhimani.com/",
        "article_pattern": ".com/",
        "language": "ml",
        "section": "india",
        "stop_markers": ["കൂടുതൽ വായിക്കുക", "Advertisement", "©", "deshabhimani.com"],
        "title_keywords": ["ഇന്ത്യ", "ദേശീയ", "കേന്ദ്ര", "സുപ്രീം കോടതി", "BJP", "മോദി"]
    },
    # ---------------- PUNJABI ----------------
    "punjabi_jagran": {
        "source": "jagran-punjabi",
        "base_url": "https://www.jagran.com/punjab",
        "article_pattern": ".html",
        "language": "pa",
        "section": "punjab",
        "stop_markers": [
            "ਹੋਰ ਪੜ੍ਹੋ",
            "Advertisement",
            "©",
            "jagran.com"
        ]
    },
    "punjabi_punjabkesari": {
    "source": "punjab-kesari",
    "base_url": "https://www.punjabkesari.in/punjab",
    "article_pattern": "/punjab/",
    "language": "pa",
    "section": "punjab-politics",
    "stop_markers": [
        "ਹੋਰ ਪੜ੍ਹੋ",
        "Advertisement",
        "©",
        "punjabkesari.in",
        "ਸੰਬੰਧਿਤ ਖ਼ਬਰਾਂ"
    ]
},
    
"assamese_pratidintime": {
    "source": "pratidin-time",
    "base_url": "https://pratidintime.com/category/assam",
    "article_pattern": "/",
    "language": "as",
    "section": "assam",
    "stop_markers": [
        "আৰু পঢ়ক",
        "Advertisement",
        "©",
        "Pratidin Time",
        "সম্পৰ্কীয় খবৰ"
    ]
},

    # ---------------- ENGLISH ----------------
"english_thehindu": {
        "source": "the-hindu",
        "base_url": "https://www.thehindu.com/news/national/",
        "article_pattern": ".ece",
        "language": "en",
        "section": "india",
        "stop_markers": ["READ MORE", "Advertisement", "©", "thehindu.com"]
    }
}


# ------------------ HELPER FUNCTIONS ------------------

def fetch(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code == 200:
            return r.text
    except Exception:
        return None
    return None


def gather_article_links(html, paper):
    soup = BeautifulSoup(html, "html.parser")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if paper["article_pattern"] in href:
            links.add(urljoin(paper["base_url"], href))

    return list(links)


def extract_body(soup, stop_markers):
    texts = []

    for p in soup.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt:
            continue

        # HARD STOP on footer
        if any(marker in txt for marker in stop_markers):
            break

        if len(txt) > 30:
            texts.append(txt)

    return "\n\n".join(texts).strip()


def extract_title(soup):
    og = soup.find("meta", property="og:title")
    if og and og.get("content"):
        return og["content"].strip()

    if soup.title:
        return soup.title.get_text(strip=True)

    return ""


def scrape_article(url, paper):
    html = fetch(url)
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")

    body = extract_body(soup, paper["stop_markers"])
    if not body:
        return None

    title = extract_title(soup)

    return {
        "url": url,
        "title": title,
        "date": "",
        "author": "",
        "body": body,
        "tags": [],
        "source": paper["source"],
        "language": paper["language"],
        "section": paper["section"]
    }


def save_jsonl(filename, record):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# ------------------ MAIN ------------------

def main():
    paper = NEWSPAPERS["punjabi_punjabkesari"]

    print(f"[INFO] Fetching homepage: {paper['base_url']}")
    html = fetch(paper["base_url"])
    if not html:
        print("[ERROR] Failed to fetch homepage")
        return

    links = gather_article_links(html, paper)
    print(f"[INFO] Found {len(links)} article links")

    collected = 0

    for url in links:
        if collected >= MAX_ARTICLES:
            print("[INFO] Reached max article limit.")
            break

        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

        try:
            article = scrape_article(url, paper)
            if article:
                save_jsonl(OUT_FILE, article)
                collected += 1
                print(f"[SAVED] {article['title'][:60]}")
            else:
                print(f"[SKIP] {url}")

        except Exception as e:
            print(f"[ERROR] {url} -> {e}")

    print(f"\nDone. Collected {collected} articles.")
    print(f"Output file: {OUT_FILE}")


if __name__ == "__main__":
    main()

[INFO] Fetching homepage: https://www.punjabkesari.in/punjab
[ERROR] Failed to fetch homepage
